# CVPR 2026 MSLR Track 2 — Score Maximizer V2
### V1 Score: 0.8478 → Target: 0.90+

**What V2 adds on top of V1 checkpoints:**

| # | Technique | Expected Boost |
|---|-----------|----------------|
| 1 | **6-channel input** (RTM + pseudo-RDM via FFT) | +2-3% — adds velocity/Doppler info |
| 2 | **Diverse new backbones** (Swin-S, ViT-S, MaxViT-T) | +2-3% — different inductive biases = better ensemble |
| 3 | **Enhanced 4-view TTA** (orig + time-flip + freq-flip + both) | +0.5-1% |
| 4 | **Accuracy-weighted ensemble** across all V1+V2 models | +0.5-1% |
| 5 | **Pseudo-labeling** with high-confidence test predictions | +1-2% |
| 6 | **Progressive resolution** fine-tuning (224→288px) | +0.5-1% |

**Setup:**
1. Upload your V1 notebook output as a Kaggle dataset (contains the 10 `.pt` checkpoint files)
2. Attach the competition dataset
3. Set `MODE = 'compete'` and run all cells

In [ ]:
# ============================================================
# CELL 1: Environment Setup
# ============================================================
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

pip_install('timm')
pip_install('einops')

import torch
print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ============================================================
# CELL 2: Imports & V2 Configuration
# ============================================================
import os, csv, time, math, random, warnings, gc, glob
from pathlib import Path
from collections import defaultdict
from copy import deepcopy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

warnings.filterwarnings('ignore')

# ---- MODE: Set to 'compete' BEFORE Kaggle submission ----
MODE = 'complete'

class CFG:
    seed        = 42
    device      = 'cuda' if torch.cuda.is_available() else 'cpu'
    num_classes = 126
    num_workers = 0 if MODE == 'debug' else 2

    # Data
    img_size      = 224
    finetune_size = 288      # progressive resize in final epochs
    max_time      = 48

    # Training
    num_folds       = 5 if MODE == 'compete' else 2
    train_folds     = list(range(num_folds))
    label_smoothing = 0.1
    weight_decay    = 0.05
    grad_clip       = 1.0
    use_amp         = True
    warmup_epochs   = 3

    # Augmentation (stronger than V1)
    mixup_alpha   = 0.4
    cutmix_alpha  = 1.0
    mix_prob      = 0.6      # up from 0.5
    noise_std     = 0.025    # slightly higher
    freq_mask     = 35       # wider
    time_mask     = 10       # wider
    channel_drop  = 0.1      # randomly zero a channel

    # SWA
    swa_frac  = 0.75         # earlier SWA
    swa_lr    = 1e-5

    # Enhanced TTA
    tta_views = 4  # orig + time-flip + freq-flip + both-flips

    # Pseudo-labeling
    pseudo_threshold = 0.92
    pseudo_epochs    = 15
    pseudo_lr        = 5e-5

    # V2 model zoo: diverse architectures, 6-channel input
    if MODE == 'debug':
        v2_models = [
            dict(name='efficientnet_b0', epochs=3, lr=3e-4, bs=16,
                 in_chans=6, finetune_epochs=0),
        ]
    else:
        v2_models = [
            dict(name='swin_small_patch4_window7_224.ms_in22k_ft_in1k',
                 epochs=40, lr=2e-4, bs=32, in_chans=6, finetune_epochs=5),
            dict(name='vit_small_patch16_224.augreg_in21k_ft_in1k',
                 epochs=40, lr=1e-4, bs=48, in_chans=6, finetune_epochs=5),
            dict(name='maxvit_tiny_tf_224.in1k',
                 epochs=40, lr=2e-4, bs=32, in_chans=6, finetune_epochs=5),
        ]


def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.benchmark = True

set_seed(CFG.seed)
print(f'MODE={MODE}, device={CFG.device}, folds={CFG.num_folds}')
print(f'V2 Models: {[m["name"] for m in CFG.v2_models]}')

In [ ]:
# ============================================================
# CELL 3: Auto-detect data paths + V1 checkpoints
# ============================================================
def find_data():
    for p in [
        # ---- Your Kaggle dataset (shujon53/cvpr-mslr-track-2) ----
        Path('/kaggle/input/datasets/shujon53/cvpr-mslr-track-2'),
        # ---- Fallbacks ----
        Path('/kaggle/input/2st-multimodal-italian-sign-language-rec'),
        Path('/kaggle/input/cvpr-mslr-2026-track-2'),
        Path('/kaggle/input/competitions/cvpr-mslr-2026-track-2'),
        Path('d:/Current-Research/CVPR2026-SignEval/cvpr-mslr-2026-track-2'),
        Path('../cvpr-mslr-2026-track-2'),
    ]:
        if (p / 'train').exists():
            print(f'  Competition data found at: {p}')
            return p
    raise FileNotFoundError('Competition dataset not found!')


def find_v1_checkpoints():
    """Find V1 .pt checkpoints.
    Kaggle dataset: shujon53/v1-ckpt-dir — .pt files are flat in the root.
    """
    search_dirs = [
        Path('/kaggle/input/datasets/shujon53/v1-ckpt-dir'),  # your uploaded dataset
        Path('/kaggle/input/v1-ckpt-dir'),                    # Kaggle slug fallback
        Path('d:/Current-Research/CVPR2026-SignEval/score_maximizer/output'),
        Path('./output'),
    ]
    v1_paths = []
    for d in search_dirs:
        if d.exists():
            found = sorted(d.rglob('best_*.pt'))
            if found:
                v1_paths.extend(found)
                print(f'  V1 checkpoints found in: {d}')
                break  # stop at first directory with checkpoints
    return v1_paths


DATA_ROOT = find_data()
TRAIN_DIR = DATA_ROOT / 'train'
VAL_DIR   = DATA_ROOT / 'val'
OUT       = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./output_v2')
OUT.mkdir(parents=True, exist_ok=True)

V1_CKPTS = find_v1_checkpoints()

print(f'\nData root:  {DATA_ROOT}')
print(f'Train dir:  {TRAIN_DIR}  (exists={TRAIN_DIR.exists()})')
print(f'Val dir:    {VAL_DIR}    (exists={VAL_DIR.exists()})')
print(f'Output:     {OUT}')
print(f'\nV1 checkpoints ({len(V1_CKPTS)} found):')
for p in V1_CKPTS:
    print(f'  {p.name}')

In [ ]:
# ============================================================
# CELL 4: Index all samples + folds
# ============================================================
def build_train_samples(train_dir):
    samples, class_names = [], []
    class_dirs = sorted(
        [d for d in Path(train_dir).iterdir()
         if d.is_dir() and d.name.split('_')[0].isdigit()],
        key=lambda d: int(d.name.split('_')[0]))
    class_names = [d.name for d in class_dirs]
    for cd in class_dirs:
        label = int(cd.name.split('_')[0])
        for sd in sorted(cd.iterdir()):
            if sd.is_dir() and sd.name.startswith('SAMPLE_'):
                if (sd / f'{sd.name}_RTM1.npy').exists():
                    samples.append((str(sd), label))
    return samples, class_names

def build_folds(samples, n_folds, seed=42):
    rng = np.random.RandomState(seed)
    c2i = defaultdict(list)
    for i, (_, l) in enumerate(samples):
        c2i[l].append(i)
    folds = [[] for _ in range(n_folds)]
    for cls in sorted(c2i):
        idxs = c2i[cls].copy()
        rng.shuffle(idxs)
        for i, idx in enumerate(idxs):
            folds[i % n_folds].append(idx)
    return folds

def build_test_samples(val_dir):
    samples = []
    for d in sorted(Path(val_dir).iterdir()):
        if d.is_dir() and d.name.startswith('SAMPLE_'):
            if (d / f'{d.name}_RTM1.npy').exists():
                samples.append((str(d), d.name))
    return samples

all_train_samples, class_names = build_train_samples(TRAIN_DIR)
fold_indices = build_folds(all_train_samples, CFG.num_folds, CFG.seed)
test_samples = build_test_samples(VAL_DIR)

print(f'Training: {len(all_train_samples)} samples, {len(class_names)} classes')
print(f'Test:     {len(test_samples)} samples')
print(f'Folds:    {[len(f) for f in fold_indices]}')

In [ ]:
# ============================================================
# CELL 5: Enhanced Dataset — 6-channel (RTM + pseudo-RDM)
# ============================================================
class RTMDatasetV2(Dataset):
    """
    V2: 6-channel input = 3 RTM channels + 3 pseudo-RDM channels (FFT along time).
    The FFT gives velocity/Doppler information that pure RTM misses.
    """
    def __init__(self, sample_list, img_size=224, max_T=48,
                 augment=False, use_rdm=True):
        self.samples  = sample_list
        self.img_size = img_size
        self.max_T    = max_T
        self.augment  = augment
        self.use_rdm  = use_rdm

    def __len__(self):
        return len(self.samples)

    def _load(self, sample_dir):
        sd = Path(sample_dir)
        sid = sd.name
        rtms = []
        for i in range(1, 4):
            arr = np.load(str(sd / f'{sid}_RTM{i}.npy')).astype(np.float32)
            rtms.append(arr)
        stacked = np.stack(rtms, axis=0)  # (3, T, 256)
        return stacked.transpose(0, 2, 1)  # (3, 256, T)

    def _compute_rdm(self, rtm):
        """Pseudo Range-Doppler Map via FFT along time axis."""
        rdm = np.abs(np.fft.fftshift(np.fft.fft(rtm, axis=2), axes=2))
        return rdm.astype(np.float32)

    def _normalize(self, x):
        mn, mx = x.min(), x.max()
        return (x - mn) / (mx - mn + 1e-8)

    def _pad_time(self, data):
        C, H, T = data.shape
        if T >= self.max_T:
            s = (T - self.max_T) // 2
            return data[:, :, s:s + self.max_T]
        pad = self.max_T - T
        pl, pr = pad // 2, pad - pad // 2
        return np.pad(data, ((0, 0), (0, 0), (pl, pr)), mode='constant')

    def _resize(self, t):
        return F.interpolate(
            t.unsqueeze(0),
            size=(self.img_size, self.img_size),
            mode='bilinear', align_corners=False
        ).squeeze(0)

    def _augment_np(self, data):
        if random.random() < 0.3:
            data = data[:, :, ::-1].copy()
        shift = random.randint(-4, 4)
        if shift:
            data = np.roll(data, shift, axis=2)
        if random.random() < 0.5:
            data = np.clip(
                data + np.random.randn(*data.shape).astype(np.float32) * CFG.noise_std,
                0, 1)
        if random.random() < CFG.channel_drop:
            ch = random.randint(0, data.shape[0] - 1)
            data[ch] = 0.0
        return data

    def _spec_augment(self, img):
        C, H, W = img.shape
        if random.random() < 0.6:
            f = random.randint(1, CFG.freq_mask)
            f0 = random.randint(0, max(0, H - f))
            img[:, f0:f0+f, :] = 0
        if random.random() < 0.6:
            t = random.randint(1, CFG.time_mask)
            t0 = random.randint(0, max(0, W - t))
            img[:, :, t0:t0+t] = 0
        if random.random() < 0.4:
            f = random.randint(1, CFG.freq_mask // 2)
            f0 = random.randint(0, max(0, H - f))
            img[:, f0:f0+f, :] = 0
        if random.random() < 0.3:
            t = random.randint(1, CFG.time_mask // 2)
            t0 = random.randint(0, max(0, W - t))
            img[:, :, t0:t0+t] = 0
        return img

    def __getitem__(self, idx):
        sd, label_or_id = self.samples[idx]
        rtm = self._load(sd)
        if self.use_rdm:
            rdm = self._compute_rdm(rtm)
            rtm = self._normalize(rtm)
            rdm = self._normalize(rdm)
            data = np.concatenate([rtm, rdm], axis=0)  # (6, 256, T)
        else:
            data = self._normalize(rtm)
        data = self._pad_time(data)
        if self.augment:
            data = self._augment_np(data)
        img = self._resize(torch.from_numpy(data.copy()).float())
        if self.augment:
            img = self._spec_augment(img)
        return img, label_or_id


# Sanity check
_ds = RTMDatasetV2(all_train_samples[:4], img_size=CFG.img_size,
                   max_T=CFG.max_time, augment=True, use_rdm=True)
img0, lbl0 = _ds[0]
print(f'V2 Dataset: shape={img0.shape}, range=[{img0.min():.3f}, {img0.max():.3f}]')
assert img0.shape[0] == 6, f'Expected 6 channels, got {img0.shape[0]}'
print('6-channel (RTM+RDM) sanity check PASSED ✓')
del _ds

In [ ]:
# ============================================================
# CELL 6: Mixup / CutMix + Model builder + Training functions
# ============================================================

# --- Mixup / CutMix ---
def mixup_data(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    _, _, H, W = x.shape
    r = np.sqrt(1 - lam)
    ch, cw = int(H * r), int(W * r)
    cy, cx = random.randint(0, H), random.randint(0, W)
    y1, y2 = max(0, cy - ch//2), min(H, cy + ch//2)
    x1, x2 = max(0, cx - cw//2), min(W, cx + cw//2)
    xc = x.clone()
    xc[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1 - (y2-y1) * (x2-x1) / (H * W)
    return xc, y, y[idx], lam

def mix_criterion(crit, pred, ya, yb, lam):
    return lam * crit(pred, ya) + (1 - lam) * crit(pred, yb)

# --- Model builder ---
def build_model(name, num_classes=126, pretrained=True, in_chans=3):
    m = timm.create_model(
        name, pretrained=pretrained, num_classes=num_classes,
        in_chans=in_chans, drop_rate=0.3, drop_path_rate=0.2,
    )
    n = sum(p.numel() for p in m.parameters()) / 1e6
    print(f'  Built {name}: {n:.1f}M params, in_chans={in_chans}')
    return m

# --- Positional embedding interpolation for resolution changes ---
def interpolate_pos_embed(state_dict, model):
    """Resize ViT absolute positional embeddings when changing resolution."""
    model_sd = model.state_dict()
    for k in list(state_dict.keys()):
        if k not in model_sd:
            continue
        if state_dict[k].shape != model_sd[k].shape:
            if 'pos_embed' in k and state_dict[k].dim() == 3:
                old = state_dict[k]
                cls_tok = old[:, :1, :]
                pos_tok = old[:, 1:, :]
                gs_old = int(pos_tok.shape[1] ** 0.5)
                gs_new = int((model_sd[k].shape[1] - 1) ** 0.5)
                print(f'      Interpolating {k}: grid {gs_old}→{gs_new}')
                D = pos_tok.shape[-1]
                pos_tok = pos_tok.reshape(1, gs_old, gs_old, D).permute(0, 3, 1, 2).float()
                pos_tok = F.interpolate(pos_tok, size=(gs_new, gs_new),
                                        mode='bicubic', align_corners=False)
                pos_tok = pos_tok.permute(0, 2, 3, 1).reshape(1, gs_new * gs_new, D)
                state_dict[k] = torch.cat([cls_tok, pos_tok], dim=1)
            else:
                # Other shape mismatches: drop key and let strict=False handle it
                print(f'      Dropping mismatched key: {k} '
                      f'({list(state_dict[k].shape)} vs {list(model_sd[k].shape)})')
                del state_dict[k]
    return state_dict

def load_checkpoint_into_model(model, state_dict):
    """Load state dict with automatic pos_embed interpolation."""
    state_dict = interpolate_pos_embed(dict(state_dict), model)
    msg = model.load_state_dict(state_dict, strict=False)
    if msg.missing_keys:
        print(f'      Missing keys: {len(msg.missing_keys)}')
    if msg.unexpected_keys:
        print(f'      Unexpected keys: {len(msg.unexpected_keys)}')

def build_model_at_size(name, img_size, num_classes=126, in_chans=3,
                        drop_rate=0.0, drop_path_rate=0.0):
    """Build a model at a specific input resolution (needed for ViT pos_embed)."""
    try:
        m = timm.create_model(name, pretrained=False, num_classes=num_classes,
                              in_chans=in_chans, drop_rate=drop_rate,
                              drop_path_rate=drop_path_rate, img_size=img_size)
    except TypeError:
        m = timm.create_model(name, pretrained=False, num_classes=num_classes,
                              in_chans=in_chans, drop_rate=drop_rate,
                              drop_path_rate=drop_path_rate)
    return m

# --- LR scheduler ---
def cosine_lr(optimizer, warmup, total, min_frac=0.01):
    def fn(ep):
        if ep < warmup:
            return (ep + 1) / warmup
        prog = (ep - warmup) / max(1, total - warmup)
        return min_frac + 0.5 * (1 - min_frac) * (1 + math.cos(math.pi * prog))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, fn)

# --- Train one epoch ---
def train_one_epoch(model, loader, opt, sched, scaler, crit, epoch, total_ep):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs   = imgs.to(CFG.device, non_blocking=True)
        labels = labels.to(CFG.device, non_blocking=True)
        do_mix = random.random() < CFG.mix_prob and epoch < total_ep - 3
        if do_mix:
            if random.random() < 0.5:
                imgs, ya, yb, lam = mixup_data(imgs, labels, CFG.mixup_alpha)
            else:
                imgs, ya, yb, lam = cutmix_data(imgs, labels, CFG.cutmix_alpha)
        with autocast(enabled=CFG.use_amp):
            logits = model(imgs)
            loss = mix_criterion(crit, logits, ya, yb, lam) if do_mix else crit(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
        loss_sum += loss.item()
        if not do_mix:
            correct += (logits.argmax(1) == labels).sum().item()
            total   += labels.size(0)
    sched.step()
    return loss_sum / max(len(loader), 1), correct / max(total, 1) * 100

# --- Validate ---
@torch.no_grad()
def validate(model, loader):
    model.eval()
    logits_all, labels_all = [], []
    for imgs, labels in loader:
        imgs = imgs.to(CFG.device, non_blocking=True)
        with autocast(enabled=CFG.use_amp):
            logits_all.append(model(imgs).cpu())
        labels_all.append(labels)
    logits_all = torch.cat(logits_all)
    labels_all = torch.cat(labels_all)
    top1 = (logits_all.argmax(1) == labels_all).float().mean().item() * 100
    _, t5 = logits_all.topk(5, 1)
    top5 = t5.eq(labels_all.unsqueeze(1)).any(1).float().mean().item() * 100
    return top1, top5

print('Training functions ready (with pos_embed interpolation)')

In [ ]:
# ============================================================
# CELL 7: Enhanced TTA — 4 views
# ============================================================
@torch.no_grad()
def predict_enhanced_tta(model, loader, n_views=4):
    """
    4-view TTA:
      0: original
      1: time-flip (horizontal flip)
      2: freq-flip (vertical flip)
      3: both flips (180° rotation)
    """
    model.eval()
    probs_list, ids_list = [], []

    for imgs, sids in loader:
        imgs = imgs.to(CFG.device, non_blocking=True)
        views = [imgs]
        if n_views >= 2:
            views.append(torch.flip(imgs, [3]))       # time flip
        if n_views >= 3:
            views.append(torch.flip(imgs, [2]))       # freq flip
        if n_views >= 4:
            views.append(torch.flip(imgs, [2, 3]))    # both

        p_sum = None
        for v in views:
            with autocast(enabled=CFG.use_amp):
                p = F.softmax(model(v), dim=1)
            p_sum = p if p_sum is None else p_sum + p

        probs_list.append((p_sum / len(views)).cpu())
        ids_list.extend(sids if isinstance(sids[0], str) else sids.tolist())

    return torch.cat(probs_list), ids_list

print('Enhanced TTA ready (4 views)')

## Phase 1: Quick Boost from V1 Checkpoints (No Retraining)
Re-run your existing V1 models with **enhanced 4-view TTA** and **accuracy-weighted ensemble** — this alone should give ~0.5-1% boost over V1.

In [ ]:
# ============================================================
# CELL 8: PHASE 1 — Enhanced inference from V1 checkpoints
# ============================================================
print('=' * 70)
print('PHASE 1: Enhanced TTA + weighted ensemble from V1 checkpoints')
print('=' * 70)

# 3-channel dataset for V1 models
test_ds_3ch = RTMDatasetV2(test_samples, img_size=224,
                            max_T=CFG.max_time, augment=False, use_rdm=False)
test_loader_3ch = DataLoader(test_ds_3ch, batch_size=96, shuffle=False,
                              num_workers=CFG.num_workers, pin_memory=True)

v1_probs   = None
v1_weights = []
sample_ids = None

if len(V1_CKPTS) > 0:
    for i, path in enumerate(V1_CKPTS):
        ckpt = torch.load(path, map_location='cpu', weights_only=False)
        # Handle both V1 save formats
        mname = ckpt.get('name', ckpt.get('model_name', 'tf_efficientnetv2_s.in21k_ft_in1k'))
        acc   = ckpt.get('acc', ckpt.get('val_top1', 80.0))
        state = ckpt.get('model', ckpt.get('model_state_dict'))
        fold  = ckpt.get('fold', '?')
        print(f'  [{i+1}/{len(V1_CKPTS)}] {mname} fold={fold} val={acc:.1f}%')

        model = timm.create_model(mname, pretrained=False, num_classes=CFG.num_classes,
                                  in_chans=3, drop_rate=0, drop_path_rate=0)
        model.load_state_dict(state)
        model = model.to(CFG.device)

        probs, ids = predict_enhanced_tta(model, test_loader_3ch, n_views=CFG.tta_views)
        w = acc / 100.0

        if v1_probs is None:
            v1_probs = probs * w
            sample_ids = ids
        else:
            v1_probs += probs * w
        v1_weights.append(w)

        del model, ckpt
        gc.collect(); torch.cuda.empty_cache()

    v1_probs /= sum(v1_weights)
    v1_preds = v1_probs.argmax(dim=1).numpy()
    print(f'\nPhase 1: {len(v1_preds)} predictions, '
          f'{len(np.unique(v1_preds))}/{CFG.num_classes} classes')
    print(f'V1 total weight: {sum(v1_weights):.2f} from {len(v1_weights)} models')
else:
    print('No V1 checkpoints found — will train from scratch in Phase 2')
    v1_probs = None

del test_ds_3ch, test_loader_3ch
gc.collect(); torch.cuda.empty_cache()

## Phase 2: Train New Diverse Backbones (6-channel RTM+RDM)
Train Swin-S, ViT-S, and MaxViT-T with 6-channel input. These have **different inductive biases** from V1's CNNs, making them excellent ensemble partners.

In [ ]:
# ============================================================
# CELL 9: PHASE 2 — Train diverse V2 models with 6-channel input
# ============================================================
print('=' * 70)
print('PHASE 2: Training diverse backbones with 6-channel RTM+RDM')
print('=' * 70)

v2_model_paths = []
v2_fold_accs   = []

for mcfg in CFG.v2_models:
    mname    = mcfg['name']
    epochs   = mcfg['epochs']
    lr       = mcfg['lr']
    bs       = mcfg['bs']
    in_chans = mcfg['in_chans']
    ft_ep    = mcfg['finetune_epochs']
    swa_start = int(epochs * CFG.swa_frac)

    print(f'\n{"─"*60}')
    print(f'MODEL: {mname}')
    print(f'epochs={epochs}+{ft_ep}ft | lr={lr} | bs={bs} | {in_chans}ch | SWA@ep{swa_start}')
    print(f'{"─"*60}')

    for fold in CFG.train_folds:
        t_fold = time.time()
        print(f'\n  Fold {fold}/{CFG.num_folds-1}')

        val_idx   = fold_indices[fold]
        train_idx = [i for f in range(CFG.num_folds) if f != fold for i in fold_indices[f]]
        train_samps = [all_train_samples[i] for i in train_idx]
        val_samps   = [all_train_samples[i] for i in val_idx]
        print(f'    Train: {len(train_samps)}, Val: {len(val_samps)}')

        use_rdm = (in_chans == 6)
        train_ds = RTMDatasetV2(train_samps, img_size=CFG.img_size,
                                max_T=CFG.max_time, augment=True, use_rdm=use_rdm)
        val_ds   = RTMDatasetV2(val_samps, img_size=CFG.img_size,
                                max_T=CFG.max_time, augment=False, use_rdm=use_rdm)
        train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True,
                                  num_workers=CFG.num_workers, pin_memory=True, drop_last=True)
        val_loader   = DataLoader(val_ds, batch_size=bs*2, shuffle=False,
                                  num_workers=CFG.num_workers, pin_memory=True)

        model = build_model(mname, in_chans=in_chans).to(CFG.device)
        opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=CFG.weight_decay)
        sched = cosine_lr(opt, CFG.warmup_epochs, epochs)
        scaler = GradScaler(enabled=CFG.use_amp)
        crit  = nn.CrossEntropyLoss(label_smoothing=CFG.label_smoothing)

        swa_model = torch.optim.swa_utils.AveragedModel(model)
        swa_sched = torch.optim.swa_utils.SWALR(opt, swa_lr=CFG.swa_lr)

        best_acc  = 0.0
        safe_name = mname.replace("/", "_").replace(".", "_")
        best_path = OUT / f'best_v2_{safe_name}_f{fold}.pt'

        # ---- Main training loop ----
        for ep in range(epochs):
            loss, tacc = train_one_epoch(model, train_loader, opt, sched,
                                         scaler, crit, ep, epochs)
            if ep >= swa_start:
                swa_model.update_parameters(model)
                swa_sched.step()

            val1, val5 = validate(model, val_loader)
            if (ep+1) % 10 == 0 or val1 > best_acc:
                print(f'    Ep {ep:3d} | loss={loss:.3f} | val={val1:.1f}%/{val5:.1f}% | '
                      f'lr={opt.param_groups[0]["lr"]:.1e}')
            if val1 > best_acc:
                best_acc = val1
                torch.save({
                    'model': model.state_dict(), 'name': mname, 'acc': val1,
                    'fold': fold, 'in_chans': in_chans, 'img_size': CFG.img_size,
                }, best_path)
                print(f'      >>> New best: {best_acc:.2f}%')

        # ---- SWA finalize ----
        try:
            torch.optim.swa_utils.update_bn(train_loader, swa_model, device=CFG.device)
            swa_acc, _ = validate(swa_model, val_loader)
            print(f'    SWA val: {swa_acc:.1f}% (best ckpt: {best_acc:.1f}%)')
            if swa_acc > best_acc:
                best_acc = swa_acc
                torch.save({
                    'model': swa_model.module.state_dict(), 'name': mname,
                    'acc': swa_acc, 'fold': fold, 'in_chans': in_chans,
                    'img_size': CFG.img_size,
                }, best_path)
                print(f'      >>> SWA saved!')
        except Exception as e:
            print(f'    SWA BN failed: {e}')

        # ---- Progressive resolution fine-tuning (224→288) ----
        if ft_ep > 0 and CFG.finetune_size > CFG.img_size:
            print(f'    Fine-tuning at {CFG.finetune_size}px for {ft_ep} epochs...')

            # Rebuild model at new resolution (handles ViT pos_embed resize)
            del model
            gc.collect(); torch.cuda.empty_cache()
            model = build_model_at_size(
                mname, img_size=CFG.finetune_size, num_classes=CFG.num_classes,
                in_chans=in_chans, drop_rate=0.3, drop_path_rate=0.2,
            ).to(CFG.device)

            ck = torch.load(best_path, map_location='cpu', weights_only=False)
            # Interpolates pos_embed if needed (e.g., ViT 224→288)
            load_checkpoint_into_model(model, ck['model'])
            print(f'    Rebuilt model at {CFG.finetune_size}px, loaded weights')

            ft_bs = max(bs // 2, 8)
            ft_train_ds = RTMDatasetV2(train_samps, img_size=CFG.finetune_size,
                                       max_T=CFG.max_time, augment=True, use_rdm=use_rdm)
            ft_val_ds   = RTMDatasetV2(val_samps, img_size=CFG.finetune_size,
                                       max_T=CFG.max_time, augment=False, use_rdm=use_rdm)
            ft_train_loader = DataLoader(ft_train_ds, batch_size=ft_bs, shuffle=True,
                                         num_workers=CFG.num_workers, pin_memory=True, drop_last=True)
            ft_val_loader   = DataLoader(ft_val_ds, batch_size=ft_bs*2, shuffle=False,
                                         num_workers=CFG.num_workers, pin_memory=True)

            ft_opt   = torch.optim.AdamW(model.parameters(), lr=lr * 0.1,
                                          weight_decay=CFG.weight_decay)
            ft_sched = cosine_lr(ft_opt, 1, ft_ep)
            ft_scaler = GradScaler(enabled=CFG.use_amp)

            for ep in range(ft_ep):
                loss, _ = train_one_epoch(model, ft_train_loader, ft_opt,
                                          ft_sched, ft_scaler, crit, ep, ft_ep)
                val1, val5 = validate(model, ft_val_loader)
                print(f'      FT Ep {ep} | loss={loss:.3f} | val={val1:.1f}%/{val5:.1f}%')
                if val1 > best_acc:
                    best_acc = val1
                    torch.save({
                        'model': model.state_dict(), 'name': mname, 'acc': val1,
                        'fold': fold, 'in_chans': in_chans,
                        'img_size': CFG.finetune_size,
                    }, best_path)
                    print(f'        >>> FT best: {best_acc:.2f}%')

            del ft_train_ds, ft_val_ds, ft_train_loader, ft_val_loader, ft_opt

        v2_model_paths.append(str(best_path))
        v2_fold_accs.append(best_acc)
        dt = time.time() - t_fold
        print(f'    Fold {fold} done: {best_acc:.2f}% in {dt/60:.1f}min')

        del model, swa_model, opt, sched, scaler
        gc.collect(); torch.cuda.empty_cache()

print(f'\nV2 fold accs: {[f"{a:.1f}%" for a in v2_fold_accs]}')
if v2_fold_accs:
    print(f'V2 mean: {np.mean(v2_fold_accs):.2f}% ± {np.std(v2_fold_accs):.2f}%')

## Phase 3: Mega Ensemble (V1 + V2, accuracy-weighted)
Combine all V1 (3ch, CNN) and V2 (6ch, Transformer) models. Each model contributes proportional to its validation accuracy — stronger models count more.

In [ ]:
# ============================================================
# CELL 10: PHASE 3 — Mega Ensemble (V1 3ch + V2 6ch)
# ============================================================
print('=' * 70)
print('PHASE 3: Mega Ensemble — V1 (3ch) + V2 (6ch), accuracy-weighted')
print('=' * 70)

# 6-channel test dataset for V2 models at base resolution
test_ds_6ch = RTMDatasetV2(test_samples, img_size=CFG.img_size,
                            max_T=CFG.max_time, augment=False, use_rdm=True)
test_loader_6ch = DataLoader(test_ds_6ch, batch_size=64, shuffle=False,
                              num_workers=CFG.num_workers, pin_memory=True)

mega_probs  = None
mega_weight = 0.0

# Add V1 probs (already computed in Phase 1)
if v1_probs is not None:
    w_v1 = sum(v1_weights)
    mega_probs = v1_probs * w_v1
    mega_weight += w_v1
    print(f'Added V1 ensemble: weight={w_v1:.2f}')

# Add V2 model predictions
for i, path in enumerate(v2_model_paths):
    if not Path(path).exists():
        continue
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    mname    = ckpt['name']
    acc      = ckpt.get('acc', 80.0)
    in_chans = ckpt.get('in_chans', 6)
    ckpt_img = ckpt.get('img_size', CFG.img_size)
    fold     = ckpt.get('fold', '?')
    print(f'  [{i+1}/{len(v2_model_paths)}] {mname} fold={fold} val={acc:.1f}% '
          f'{in_chans}ch {ckpt_img}px')

    # Use appropriate dataset based on checkpoint config
    if ckpt_img != CFG.img_size:
        ds_tmp = RTMDatasetV2(test_samples, img_size=ckpt_img,
                               max_T=CFG.max_time, augment=False, use_rdm=(in_chans == 6))
        loader_tmp = DataLoader(ds_tmp, batch_size=48, shuffle=False,
                                 num_workers=CFG.num_workers, pin_memory=True)
    else:
        loader_tmp = test_loader_6ch

    # Build model at checkpoint's resolution and load with pos_embed interpolation
    model = build_model_at_size(mname, img_size=ckpt_img, num_classes=CFG.num_classes,
                                in_chans=in_chans)
    load_checkpoint_into_model(model, ckpt['model'])
    model = model.to(CFG.device)

    probs, ids = predict_enhanced_tta(model, loader_tmp, n_views=CFG.tta_views)
    w = acc / 100.0

    if mega_probs is None:
        mega_probs = probs * w
        sample_ids = ids
    else:
        mega_probs += probs * w
    mega_weight += w

    del model, ckpt
    gc.collect(); torch.cuda.empty_cache()

mega_probs /= mega_weight
mega_preds  = mega_probs.argmax(dim=1).numpy()

print(f'\nMega-ensemble: total weight={mega_weight:.2f}')
print(f'Predictions: {len(mega_preds)} samples, '
      f'{len(np.unique(mega_preds))}/{CFG.num_classes} classes')

# Confidence analysis for pseudo-labeling
confidence = mega_probs.max(dim=1)[0].numpy()
print(f'Confidence: mean={confidence.mean():.3f}, '
      f'median={np.median(confidence):.3f}')
print(f'High-conf (>{CFG.pseudo_threshold}): '
      f'{(confidence > CFG.pseudo_threshold).sum()} / {len(confidence)}')

## Phase 4: Pseudo-Labeling (Optional but Powerful)
Use high-confidence predictions from the mega-ensemble to augment training data. This is essentially free labeled data that can push your score higher.

In [ ]:
# ============================================================
# CELL 11: PHASE 4 — Pseudo-labeling
# ============================================================
print('=' * 70)
print('PHASE 4: Pseudo-labeling — retrain with high-confidence predictions')
print('=' * 70)

# Create pseudo-labeled samples from confident test predictions
pseudo_samples = []
for i, (sd, sid) in enumerate(test_samples):
    if confidence[i] > CFG.pseudo_threshold:
        pseudo_label = int(mega_preds[i])
        pseudo_samples.append((sd, pseudo_label))

print(f'Pseudo-labeled: {len(pseudo_samples)} / {len(test_samples)} '
      f'(threshold={CFG.pseudo_threshold})')

pseudo_model_paths = []
pseudo_accs = []

if len(pseudo_samples) > 100:
    # Pick best V2 backbone for pseudo-label retraining
    if v2_fold_accs and CFG.v2_models:
        best_mcfg = CFG.v2_models[0]
        n_folds_per = CFG.num_folds
        best_mean = 0
        for mi, mc in enumerate(CFG.v2_models):
            start = mi * n_folds_per
            end = start + n_folds_per
            fold_accs = v2_fold_accs[start:end] if end <= len(v2_fold_accs) else []
            if fold_accs:
                m = np.mean(fold_accs)
                if m > best_mean:
                    best_mean, best_mcfg = m, mc
        print(f'Retraining: {best_mcfg["name"]} (best mean: {best_mean:.1f}%)')
    else:
        best_mcfg = CFG.v2_models[0] if CFG.v2_models else dict(
            name='efficientnet_b0', in_chans=6, bs=32)

    mname    = best_mcfg['name']
    in_chans = best_mcfg.get('in_chans', 6)
    bs       = best_mcfg.get('bs', 32)
    use_rdm  = (in_chans == 6)

    for fold in CFG.train_folds:
        print(f'\n  Pseudo fold {fold}')
        val_idx   = fold_indices[fold]
        train_idx = [i for f in range(CFG.num_folds) if f != fold for i in fold_indices[f]]
        train_samps = [all_train_samples[i] for i in train_idx]
        val_samps   = [all_train_samples[i] for i in val_idx]

        # Combine real + pseudo samples
        combined = train_samps + pseudo_samples
        print(f'    Train: {len(train_samps)} real + {len(pseudo_samples)} pseudo = {len(combined)}')

        # Pseudo-label retraining at base resolution (224) for speed
        train_ds = RTMDatasetV2(combined, img_size=CFG.img_size,
                                max_T=CFG.max_time, augment=True, use_rdm=use_rdm)
        val_ds   = RTMDatasetV2(val_samps, img_size=CFG.img_size,
                                max_T=CFG.max_time, augment=False, use_rdm=use_rdm)
        train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True,
                                  num_workers=CFG.num_workers, pin_memory=True, drop_last=True)
        val_loader   = DataLoader(val_ds, batch_size=bs*2, shuffle=False,
                                  num_workers=CFG.num_workers, pin_memory=True)

        # Initialize from best V2 checkpoint (may be 224 or 288px)
        safe_name = mname.replace("/", "_").replace(".", "_")
        v2_ckpt_path = OUT / f'best_v2_{safe_name}_f{fold}.pt'

        # Build model at base resolution for pseudo-label training
        model = build_model(mname, in_chans=in_chans, pretrained=False).to(CFG.device)
        if v2_ckpt_path.exists():
            ck = torch.load(v2_ckpt_path, map_location='cpu', weights_only=False)
            # Handles pos_embed 288→224 interpolation if checkpoint was fine-tuned
            load_checkpoint_into_model(model, ck['model'])
            print(f'    Loaded: {v2_ckpt_path.name}')

        opt    = torch.optim.AdamW(model.parameters(), lr=CFG.pseudo_lr,
                                    weight_decay=CFG.weight_decay)
        sched  = cosine_lr(opt, 1, CFG.pseudo_epochs)
        scaler = GradScaler(enabled=CFG.use_amp)
        crit   = nn.CrossEntropyLoss(label_smoothing=0.15)

        best_acc = 0.0
        best_path = OUT / f'best_pseudo_{safe_name}_f{fold}.pt'

        for ep in range(CFG.pseudo_epochs):
            loss, _ = train_one_epoch(model, train_loader, opt, sched,
                                       scaler, crit, ep, CFG.pseudo_epochs)
            val1, val5 = validate(model, val_loader)
            if (ep+1) % 5 == 0 or val1 > best_acc:
                print(f'    Ep {ep:3d} | loss={loss:.3f} | val={val1:.1f}%/{val5:.1f}%')
            if val1 > best_acc:
                best_acc = val1
                torch.save({
                    'model': model.state_dict(), 'name': mname, 'acc': val1,
                    'fold': fold, 'in_chans': in_chans, 'img_size': CFG.img_size,
                }, best_path)
                print(f'      >>> Pseudo best: {best_acc:.2f}%')

        pseudo_model_paths.append(str(best_path))
        pseudo_accs.append(best_acc)

        del model, opt, sched, scaler
        gc.collect(); torch.cuda.empty_cache()

    print(f'\nPseudo fold accs: {[f"{a:.1f}%" for a in pseudo_accs]}')
    if pseudo_accs:
        print(f'Pseudo mean: {np.mean(pseudo_accs):.2f}%')
else:
    print('Not enough confident predictions — skipping pseudo-labeling')

## Phase 5: Final Ultra-Ensemble + Submission
Combine ALL models: V1 (10 models, 3ch) + V2 (15 models, 6ch) + Pseudo-labeled (5 models). Each weighted by validation accuracy.

In [ ]:
# ============================================================
# CELL 12: PHASE 5 — Final Ultra-Ensemble
# ============================================================
print('=' * 70)
print('PHASE 5: Final Ultra-Ensemble — V1 + V2 + Pseudo-labeled')
print('=' * 70)

final_probs  = None
final_weight = 0.0

# Start with V1 probs
if v1_probs is not None:
    w_v1 = sum(v1_weights) * 1.0
    final_probs = v1_probs * w_v1
    final_weight += w_v1
    print(f'V1 contribution: weight={w_v1:.2f}')

# Add all V2 + pseudo models
all_v2_paths = v2_model_paths + pseudo_model_paths
all_v2_accs  = v2_fold_accs + pseudo_accs

if all_v2_paths:
    for i, path in enumerate(all_v2_paths):
        if not Path(path).exists():
            continue
        ckpt = torch.load(path, map_location='cpu', weights_only=False)
        mname    = ckpt['name']
        acc      = ckpt.get('acc', 80.0)
        in_chans = ckpt.get('in_chans', 6)
        ckpt_img = ckpt.get('img_size', CFG.img_size)

        ds = RTMDatasetV2(test_samples, img_size=ckpt_img,
                           max_T=CFG.max_time, augment=False, use_rdm=(in_chans == 6))
        loader = DataLoader(ds, batch_size=48, shuffle=False,
                             num_workers=CFG.num_workers, pin_memory=True)

        # Build model at checkpoint's resolution and load with pos_embed interpolation
        model = build_model_at_size(mname, img_size=ckpt_img, num_classes=CFG.num_classes,
                                    in_chans=in_chans)
        load_checkpoint_into_model(model, ckpt['model'])
        model = model.to(CFG.device)

        probs, ids = predict_enhanced_tta(model, loader, n_views=CFG.tta_views)
        w = acc / 100.0
        # Pseudo models get slightly lower weight
        is_pseudo = 'pseudo' in Path(path).name
        if is_pseudo:
            w *= 0.85

        if final_probs is None:
            final_probs = probs * w
            sample_ids  = ids
        else:
            final_probs += probs * w
        final_weight += w

        tag = ' [pseudo]' if is_pseudo else ''
        print(f'  Added {mname} f{ckpt.get("fold","?")} acc={acc:.1f}% w={w:.3f}{tag}')

        del model, ckpt
        gc.collect(); torch.cuda.empty_cache()

final_probs /= final_weight
final_preds  = final_probs.argmax(dim=1).numpy()

print(f'\nFinal ensemble: total_weight={final_weight:.2f}')
print(f'Predictions: {len(final_preds)} samples, '
      f'{len(np.unique(final_preds))}/{CFG.num_classes} classes')

In [ ]:
# ============================================================
# CELL 13: Write Submission CSV
# ============================================================
import pandas as pd

rows = []
for sid, pred in zip(sample_ids, final_preds):
    nid = int(sid.replace('SAMPLE_', '')) if isinstance(sid, str) else int(sid)
    rows.append({'id': nid, 'Pred': int(pred)})
rows.sort(key=lambda r: r['id'])

sub_path = OUT / 'submission.csv'
with open(sub_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['id', 'Pred'])
    w.writeheader()
    w.writerows(rows)

df = pd.read_csv(sub_path)
print(f'\n{"="*70}')
print(f'SUBMISSION: {sub_path}')
print(f'{"="*70}')
print(f'Shape:       {df.shape}')
print(f'Pred range:  [{df.Pred.min()}, {df.Pred.max()}]')
print(f'Unique:      {df.Pred.nunique()} / {CFG.num_classes}')
print(f'Any NaN:     {df.isnull().any().any()}')
print(f'\nHead:\n{df.head(10)}')
print(f'\nClass distribution (top 10):')
print(df.Pred.value_counts().head(10))

final_conf = final_probs.max(dim=1)[0].numpy()
print(f'\nConfidence: mean={final_conf.mean():.3f}, '
      f'min={final_conf.min():.3f}, max={final_conf.max():.3f}')

---
## Done! V2 Pipeline Complete

### Expected Score Progression:
| Pipeline | Expected Score |
|----------|----------------|
| V1 baseline (your 10-model ensemble) | **0.8478** |
| + Enhanced 4-view TTA (Phase 1 only) | ~0.855 |
| + V2 6ch diverse backbones (Phases 1-3) | ~0.88-0.90 |
| + Pseudo-labeling (Phases 1-5) | **0.90-0.93** |

### Tips for Further Improvement:
1. **Add more backbones:** `swin_base_patch4_window7_224`, `eva02_small_patch14_224`
2. **Higher resolution:** Fine-tune at 336px or 384px (reduce batch size)
3. **Second pseudo-labeling round:** Use the improved model to generate new pseudo-labels
4. **Multi-scale TTA:** Also run inference at 256px and 192px, ensemble
5. **Temperature scaling:** Calibrate probabilities before ensembling

### Kaggle Setup Checklist:
- [ ] Add V1 output as dataset input
- [ ] Attach competition dataset
- [ ] Set `MODE = 'compete'`
- [ ] GPU accelerator enabled (P100 or T4)
- [ ] Internet OFF (for competition submission)